# 预期寿命分析

探索全球预期寿命的两个数据集：
- **Gapminder**（1952-2007）：country、year、population、continent、lifeExp、gdpPercap
- **世界卫生组织 (WHO) 预期寿命**（2000-2015）：193 个国家、22 项指标（死亡率、BMI、GDP、受教育程度等）

本工作簿演示了在 **Python** 和 **R** 中导入和分析 CSV 文件。

## 1. 环境准备：安装软件包并下载数据集

In [ ]:
import micropip
await micropip.install(['pandas', 'plotly'])
print('已安装 pandas + plotly')

import pyodide.http, os

datasets = {
    "gapminder.csv": "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv",
    "who_life_expectancy.csv": "https://raw.githubusercontent.com/Sid-149/Life-Expectancy-Predictor-Comparative-Analysis/main/Notebooks/Life%20Expectancy%20Data.csv"
}

os.makedirs("/shared/data", exist_ok=True)

for name, url in datasets.items():
    path = f"/shared/data/{name}"
    if os.path.exists(path):
        print(f"已存在: {path}")
    else:
        resp = await pyodide.http.pyfetch(url)
        text = await resp.string()
        with open(path, "w") as f:
            f.write(text)
        lines = text.count("\n")
        print(f"已下载 {name}: {lines} 行")

## 2. Gapminder：使用 Python 进行探索

In [ ]:
import pandas as pd

gap = pd.read_csv("/shared/data/gapminder.csv")
print(f"形状: {gap.shape}")
print(f"大洲: {sorted(gap['continent'].unique())}")
print(f"年份范围: {gap['year'].min()}-{gap['year'].max()}")
print()
gap.describe()

In [ ]:
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

# 各大洲预期寿命随时间的变化
avg = gap.groupby(['year', 'continent'])['lifeExp'].mean().reset_index()
fig = px.line(avg, x='year', y='lifeExp', color='continent',
              title='各大洲预期寿命 (1952-2007)',
              labels={'lifeExp': '预期寿命（岁）', 'year': '年份'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

In [ ]:
# 人均 GDP vs 预期寿命 (2007)，气泡大小 = 人口
g2007 = gap[gap['year'] == 2007]
fig = px.scatter(g2007, x='gdpPercap', y='lifeExp', size='pop',
                 color='continent', hover_name='country',
                 log_x=True, size_max=50,
                 title='人均 GDP vs 预期寿命 (2007)',
                 labels={'gdpPercap': '人均 GDP（对数）', 'lifeExp': '预期寿命'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 3. Gapminder：使用 R 进行探索

In [ ]:
gap <- read.csv("/shared/data/gapminder.csv")
str(gap)
summary(gap$lifeExp)

In [ ]:
# 各大洲预期寿命分布（箱线图）
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
boxplot(lifeExp ~ continent, data = gap,
        main = "各大洲预期寿命",
        xlab = "大洲", ylab = "预期寿命（岁）",
        col = c("#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"),
        border = "white")

In [ ]:
# 预期寿命提升前 10 名国家 (1952 vs 2007)
early <- gap[gap$year == 1952, c("country", "lifeExp")]
late  <- gap[gap$year == 2007, c("country", "lifeExp")]
merged <- merge(early, late, by = "country", suffixes = c("_1952", "_2007"))
merged$improvement <- merged$lifeExp_2007 - merged$lifeExp_1952
top10 <- head(merged[order(-merged$improvement), ], 10)

par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white", mar = c(5, 10, 4, 2))
barplot(top10$improvement, names.arg = top10$country,
        horiz = TRUE, las = 1,
        main = "前 10 名：预期寿命增长 (1952-2007)",
        xlab = "增长年数",
        col = "#00CC96", border = NA)

## 4. WHO 预期寿命：使用 Python 进行探索

In [ ]:
who = pd.read_csv("/shared/data/who_life_expectancy.csv")
print(f"形状: {who.shape}")
print(f"列名: {list(who.columns)}")
print(f"\n缺失值（前 5 项）:")
print(who.isnull().sum().sort_values(ascending=False).head())
print()
who.head()

In [ ]:
# 发展中国家 vs 发达国家：预先分箱的预期寿命分布
# 显式柱状图坐标在通过浏览器 Plotly 桥接渲染时表现一致。
import numpy as np
life = who.dropna(subset=['Life expectancy'])
edges = np.linspace(life['Life expectancy'].min(), life['Life expectancy'].max(), 41)
bin_width = edges[1] - edges[0]
hist_rows = []
for status, group in life.groupby('Status'):
    counts, _ = np.histogram(group['Life expectancy'], bins=edges)
    hist_rows.extend({
        'Life expectancy': float(left + bin_width / 2),
        'Count': int(count),
        'Status': status
    } for left, count in zip(edges[:-1], counts))

hist = pd.DataFrame(hist_rows)
fig = px.bar(hist, x='Life expectancy', y='Count', color='Status',
             barmode='overlay', opacity=0.7,
             title='预期寿命：发展中国家 vs 发达国家',
             labels={'Life expectancy': '预期寿命（岁）'})
fig.update_traces(width=float(bin_width * 0.92))
fig.update_layout(template='plotly_dark', bargap=0.03)
show_plotly(fig)

In [ ]:
# 受教育程度 vs 预期寿命
w2014 = who[who['Year'] == 2014].dropna(subset=['Schooling', 'Life expectancy'])
fig = px.scatter(w2014, x='Schooling', y='Life expectancy',
                 color='Status', hover_name='Country',
                 title='受教育程度 vs 预期寿命 (2014)',
                 labels={'Life expectancy': '预期寿命（岁）',
                         'Schooling': '受教育年限'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 5. WHO 预期寿命：使用 R 进行探索

In [ ]:
who <- read.csv("/shared/data/who_life_expectancy.csv")
str(who)
cat("\n国家数量:", length(unique(who$Country)))
cat("\n年份范围:", range(who$Year))

In [ ]:
# 相关性：成年人死亡率 vs 预期寿命
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
plot(who$Adult.Mortality, who$Life.expectancy,
     pch = 16, cex = 0.5,
     col = ifelse(who$Status == "Developed", "#636EFA80", "#EF553B80"),
     main = "成年人死亡率 vs 预期寿命",
     xlab = "成年人死亡率（每千人）",
     ylab = "预期寿命（岁）")
legend("topright", legend = c("发达国家", "发展中国家"),
       col = c("#636EFA", "#EF553B"), pch = 16, text.col = "white")

In [ ]:
# 简单线性模型：什么能预测预期寿命？
who_clean <- na.omit(who[, c("Life.expectancy", "Schooling",
                              "Adult.Mortality", "GDP", "BMI")])
model <- lm(Life.expectancy ~ Schooling + Adult.Mortality + log1p(GDP) + BMI,
            data = who_clean)
summary(model)

## 核心发现

- 全球预期寿命普遍上升，但各大洲之间仍存在巨大差距
- GDP 和受教育程度是预期寿命强劲的正向预测因子
- 成年人死亡率是最显著的负向预测因子
- 发展中国家的结果表现出大得多的方差